# LLM Gateway Patterns: Routing, Rate Limiting, and Cost Tracking

A gateway sits between your application and the LLM APIs, providing cross-cutting concerns: authentication, rate limiting, model routing, fallback handling, and cost tracking. For organizations using multiple models, a gateway is the control plane.

In [ ]:
from dataclasses import dataclass, field
from typing import Callable, Optional
import time, math

@dataclass
class ModelConfig:
    name: str
    provider: str
    cost_per_input_token: float
    cost_per_output_token: float
    max_tokens: int
    latency_ms: int
    priority: int = 1

@dataclass
class GatewayRequest:
    prompt: str
    max_tokens: int = 1000
    model_preference: Optional[str] = None
    require_fast: bool = False
    require_cheap: bool = False
    user_id: str = 'default'

@dataclass
class GatewayResponse:
    content: str
    model_used: str
    input_tokens: int
    output_tokens: int
    cost_usd: float
    latency_ms: float
    cached: bool = False

class LLMGateway:
    def __init__(self, models: list, rate_limits: dict = None):
        self.models = {m.name: m for m in models}
        self.rate_limits = rate_limits or {}  # user_id -> calls_per_minute
        self._call_times: dict = {}  # user_id -> list of timestamps
        self._total_cost: float = 0.0
        self._call_log: list = []

    def _check_rate_limit(self, user_id: str) -> bool:
        limit = self.rate_limits.get(user_id, self.rate_limits.get('default', 1000))
        now = time.time()
        times = self._call_times.get(user_id, [])
        times = [t for t in times if now - t < 60]  # keep last minute
        self._call_times[user_id] = times
        if len(times) >= limit:
            return False
        self._call_times[user_id].append(now)
        return True

    def _select_model(self, req: GatewayRequest) -> ModelConfig:
        if req.model_preference and req.model_preference in self.models:
            return self.models[req.model_preference]
        candidates = list(self.models.values())
        if req.require_fast:
            candidates.sort(key=lambda m: m.latency_ms)
        elif req.require_cheap:
            candidates.sort(key=lambda m: m.cost_per_input_token)
        else:
            candidates.sort(key=lambda m: m.priority)
        return candidates[0]

    def call(self, req: GatewayRequest, model_fn: Callable) -> GatewayResponse:
        if not self._check_rate_limit(req.user_id):
            raise RuntimeError(f'Rate limit exceeded for user {req.user_id}')
        model = self._select_model(req)
        start = time.time()
        try:
            response = model_fn(req.prompt, model.name)
        except Exception as e:
            # Fallback to next available model
            fallbacks = [m for m in self.models.values() if m.name != model.name]
            if fallbacks:
                model = fallbacks[0]
                response = model_fn(req.prompt, model.name)
            else:
                raise
        latency = (time.time() - start) * 1000
        input_tokens = len(req.prompt.split()) * 4 // 3
        output_tokens = len(response.split()) * 4 // 3
        cost = (input_tokens * model.cost_per_input_token +
                output_tokens * model.cost_per_output_token) / 1e6
        self._total_cost += cost
        self._call_log.append({'model': model.name, 'cost': cost, 'user': req.user_id})
        return GatewayResponse(content=response, model_used=model.name,
                                input_tokens=input_tokens, output_tokens=output_tokens,
                                cost_usd=cost, latency_ms=latency)

    def cost_report(self) -> dict:
        by_model = {}
        for call in self._call_log:
            by_model[call['model']] = by_model.get(call['model'], 0) + call['cost']
        return {'total_cost': self._total_cost, 'calls': len(self._call_log), 'by_model': by_model}

models = [
    ModelConfig('claude-haiku', 'anthropic', 0.25, 1.25, 200000, 200, priority=1),
    ModelConfig('claude-sonnet', 'anthropic', 3.0, 15.0, 200000, 600, priority=2),
    ModelConfig('claude-opus', 'anthropic', 15.0, 75.0, 200000, 1500, priority=3),
]
gw = LLMGateway(models, rate_limits={'default': 10})

rng = random.Random(42)
def mock_model(prompt, model_name):
    return f'Response from {model_name}: ' + prompt[:50] + '...'

requests = [
    GatewayRequest('Quick simple question about Python', require_fast=True),
    GatewayRequest('Complex analysis requiring high quality', model_preference='claude-opus'),
    GatewayRequest('Cost-sensitive batch processing task', require_cheap=True),
]
for req in requests:
    resp = gw.call(req, mock_model)
    print(f'Model: {resp.model_used:<20} Cost: ${resp.cost_usd:.6f}  Prompt: {req.prompt[:40]}')

report = gw.cost_report()
print(f'\nTotal cost: ${report["total_cost"]:.6f}')
print(f'By model: {report["by_model"]}')